In [2]:
import json
import requests
import pandas as pd

# The official government data API endpoint for real-time CPCB stations
# This source gives a clean distribution of state, city, station names, and exact geo-coordinates.
url = "https://api.data.gov.in/resource/3b01bcb8-0b14-4abf-b6f2-c1bfd384ba69"

# We request a large limit to grab all available stations across India in one sweep
params = {
    "api-key": "579b464db66ec23bdd000001cdd394682c7b4d585172221d3159024e", # Public data.gov.in fallback key
    "format": "json",
    "limit": "1000" 
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

try:
    print("Fetching live station list from CPCB repository...")
    response = requests.get(url, params=params, headers=headers, timeout=20)
    
    if response.status_code == 200:
        raw_data = response.json()
        records = raw_data.get("records", [])
        
        if not records:
            print("Connected, but no records returned. Checking structure...")
            # Fallback if structure changes slightly
            if "records" not in raw_data and isinstance(raw_data, list):
                records = raw_data

        parsed_stations = []
        for station in records:
            # Safely capture names and handle variations in CPCB API key names
            state = station.get("state") or station.get("stateName", "")
            city = station.get("city") or station.get("cityName", "")
            name = station.get("station") or station.get("stationName", "")
            lat = station.get("latitude") or station.get("lat")
            lng = station.get("longitude") or station.get("lon") or station.get("long")

            if lat and lng:
                parsed_stations.append({
                    "State": str(state).strip(),
                    "City": str(city).strip(),
                    "Station Name": str(name).strip(),
                    "Latitude": float(lat),
                    "Longitude": float(lng)
                })
        
        # Convert to Pandas DataFrame
        df = pd.DataFrame(parsed_stations)
        
        if not df.empty:
            # Drop rows missing coordinates and wipe duplicate entries
            df.dropna(subset=["Latitude", "Longitude"], inplace=True)
            df.drop_duplicates(subset=["State", "City", "Station Name"], inplace=True)
            df.sort_values(by=["State", "City", "Station Name"], inplace=True)
            
            # Save out to the pristine .xlsx format you need
            output_file = "cpcb_stations.xlsx"
            df.to_excel(output_file, index=False, engine="openpyxl")
            print(f"\nSuccess! Found and saved {len(df)} live CPCB stations to '{output_file}'")
        else:
            print("Failed to parse valid coordinates from the stream data.")
            
    else:
        print(f"Failed to connect. HTTP Status Code: {response.status_code}")
        print("Attempting backup extraction...")

except Exception as e:
    print(f"An error occurred: {e}")

Fetching live station list from CPCB repository...
Failed to connect. HTTP Status Code: 403
Attempting backup extraction...


In [3]:
# What's in the raw files vs the cleaned Excel?
import glob, os
raw_stations = set(os.path.basename(f).rsplit('.',1)[0].strip() 
                   for f in glob.glob(os.path.join(RAW_PATH, '*.xls*')))
excel_stations = set(station_names)

print("Raw files not in cleaned Excel (skipped):")
print(f"  {sorted(raw_stations - excel_stations - set(DROP_STATIONS))}")
print("\nDropped as intended:")
print(f"  {sorted(raw_stations & set(DROP_STATIONS))}")
print(f"\nRaw files    : {len(raw_stations)}")
print(f"Excel (clean): {len(excel_stations)}")

NameError: name 'RAW_PATH' is not defined